**GEOG5415M Programming for Spatial Data Science**

# Week 5: Using Python as a GIS <a class="tocSkip">

#### Contact: N.S.Malleson@leeds.ac.uk <a class="tocSkip">

# Part A: Getting started with spatial data

Up until now all the packages we have needed have already been installed in the default Google Colab environment. This week we need some packages that are not: contextily (for basemaps), mapclassify (for classifying data in choropleth maps) and geo_northarrow (for adding a north arrow).

To install packages from within a notebook the quickest way is to use: !pip install PACKAGENAME

- the ! character is used to execute shell commands directly from a code cell. This allows you to run commands that you would normally run in a terminal or command prompt.
- pip is a Python package installer. It's used to install and manage software packages written in Python.
- install: a command for pip that tells it to install a package
- PACKAGENAME e.g. contextily or mapclassify: examples of packages we can install

If a package is already installed, pip will simply tell you so.

In [ ]:
# Install the packages that are not in the default Colab environment.
# In Colab, run this cell at the start of every session (installed packages are lost when the session ends).
# On your own computer you only need to run it once.
!pip install contextily mapclassify
!pip install git+https://github.com/pmdscully/geo_northarrow.git

As is the custom when coding, we import the required packages at the beginning<br>
<font color='orchid'> <b>Import the packages below </font>

In [ ]:
#Import the required packages

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

# for adding basemaps
import contextily as ctx

# for adding a north arrow to maps
from geo_northarrow import add_north_arrow

# set seaborn plotting theme to white
sns.set_theme(style="white")


We have imported a new package 'geopandas'. Geopandas works like pandas but also handles spatial data. Geopandas was designed to allow people to easily handle and use spatial datasets in Python. <br>



## Reading in spatial data

Geopandas data frames work the same way as pandas data frames, but are also able to handle the spatial element of the dataset. Therefore, the code we use is very similar. We use <code>geopandas.read_file(<font color =red>file_path</font>) </code> replacing <font color =red>file_path</font> with the path (or URL) of the spatial data file you want to read (in this case .geojson files).

We are going to read the files from the GitHub repo. The urls to the data on GitHub are:
- https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/blob/main/data/week_5/West_Yorkshire_bus_stops.geojson
- https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/blob/main/data/week_5/leeds_travel_to_work_mode_distance.geojson

Remember, when we are reading in data from GitHub we need to amend the link from: '/blob/' to: '/raw/refs/heads/' in the code. 


<font color='orchid'> Run the code below to read the spatial data in </font>

In [ ]:
bus_stops =gpd.read_file('https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_5/West_Yorkshire_bus_stops.geojson')
leeds=gpd.read_file('https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_5/leeds_travel_to_work_mode_distance.geojson')


### Data Exploration
Just as we have done before we are going to explore the data by having a look at the data frames and by visualising both the data sets.

In [ ]:
# let's have a look at the bus_stops geopandas dataframe
bus_stops.head()

### West Yorkshire Bus Stops

https://datamillnorth.org/dataset/vqxk4/west-yorkshire-bus-stops

The data includes:
- 'STOPID': a unique ID for each bus stop
- 'NAME': the name of each bus stop
- 'TOWN': the town in which the bus stop is located
- '0730_0930'...'2000_2100': average number of buses per hour during the AM peak (0730-0930), inter-peak (1200-1400), PM peak (1500-1830) and evening (2000-2100) time periods.
- 'x' and 'y': coordinates (British National Grid eastings and northings, in metres)
- 'z': elevation of the bus stop
- 'PhotoF': link to a front photo of the bus stop
- 'PhotoS': link to a side photo of the bus stop

- 'geometry': the type of geometry, in this case POINT Z, followed by the coordinates of the point. The Z means each point has three numbers: x, y and z (elevation).





In [ ]:
# let's have a look at the Leeds geopandas dataframe
leeds.head()

### Leeds
LSOA level polygon file for Leeds with 2021 census data about travel to work distance and transport mode, downloaded from [geoportal](https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bfc-v10-2/about): 
- Notice this time the 'geometry' column has more than two numbers and 'POLYGON()' rather than 'POINT()'

## Simple spatial data visualisation
### Point data

In [ ]:
# Plot the bus stop data

# define the plot size and number of subplots (1 i.e. 1 plot)
f, ax = plt.subplots(1, figsize=(16, 8))
# plot the bus stops, specifying the subplot axis
bus_stops.plot(ax=ax)
# show the plot
plt.show()

### Polygon data

In [ ]:
# Plot the leeds data

# define the plot size and number of subplots (1 i.e. 1 plot)
f, ax = plt.subplots(1, figsize=(16, 8))
# plot the leeds data, specifying the subplot axis
leeds.plot(ax=ax)
# show the plot
plt.show()

<b>Note</b> the geometry column this time is made up of POLYGON data, made up of many coordinate points. <br>
Let us have a closer look at the polygon geometry. We are going to look at the row with index 3.

In [ ]:
# Let's look at the row with index 3 (the LSOA 'Leeds 009B')
leeds.loc[3,:]

In [ ]:
# Now let us look at the geometry column
# Notice the shape is quite complex with lots of edges
leeds.loc[3,'geometry']

In [ ]:
# let us look at the list of coordinates
print(leeds.loc[3,'geometry'])
# Each point is a corner of the LSOA

### geopandas.explore()

<code>.explore()</code> is a really useful function that generates an interactive Leaflet map from a GeoDataFrame. It is particularly useful for initial data exploration. The function creates:
- a navigable map
- with 'zoom in and out' functionality
- and hover over function displaying the variables in the GeoDataFrame

The <code>.explore()</code> has many customisable parameters, see the [package documentation](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.explore.html)

In [ ]:
bus_stops.explore()

In [ ]:
leeds.explore()

## Coordinate Reference Systems (CRS)
Before we map multiple layers we need to check they have the same Coordinate Reference System (CRS). We can do this using the <code>.crs</code> attribute.

This displays useful information about the CRS, such as its units and the areas it can be used in.

Each CRS has a code in the EPSG registry. The three you will meet most often are:
- EPSG:27700: British National Grid (units: metres). Use this for analysis in Great Britain.
- EPSG:4326: WGS 84 latitude and longitude (units: degrees). Used by GPS and most web data.
- EPSG:3857: Web Mercator (units: metres). Used by web maps and basemaps.

In [ ]:
leeds.crs

Look at what happens when we change the projection

In [ ]:
# NAD83 Canadian Spatial Reference System: Large and medium scale topographic mapping and engineering survey.
leeds_new_proj= leeds.to_crs(epsg=2953)

# EPSG:3851 New Zealand Geodetic Datum 2000: Spatial referencing and conformal mapping on the NZ continental shelf.
leeds_new_proj2= leeds.to_crs(epsg=3851)

In [ ]:
leeds_new_proj.crs

In [ ]:
# plot the two different new projections alongside the original projection
f, ax = plt.subplots(1,3, figsize=(30, 20))
leeds.plot(ax=ax[0])
leeds_new_proj.plot(ax=ax[1])
leeds_new_proj2.plot(ax=ax[2])
plt.show()
# Note the different scales on the axis and orientation of the LSOAs

## Layering maps
Much like other mapping software it is possible to layer maps in Python. We will plot the bus stops on top of the leeds dataset.

We need to check the CRS of both datasets is the same, so we can accurately plot the layers on top of each other.

In [ ]:
# Check the CRS are the same
print(leeds.crs, bus_stops.crs)

## Plot the bus stops and LSOAs together

In [ ]:
# plot one subplot (1 map), with dimensions 16 X 8
f, ax = plt.subplots(1, figsize=(16, 8))

# define the basemap plot it on the subplot axis
base = leeds.plot(ax=ax)

# plot the bus stops on the basemap axis, colour the bus stops red
bus_stops.plot(ax=base,color='red')

# show the map
plt.show()

As we can see we are able to plot the bus stop data (for the whole of West Yorkshire) onto the Leeds Map.

## Choropleth Mapping
Choropleth maps are maps where the polygons are coloured different shades or colours based on a value. E.g. Number of people commuting by bus per LSOA

In [ ]:
# Define plot size
f,ax = plt.subplots(1, figsize=(16,8))
# Plot the LSOAs, specifying to plot the 'Bus, minibus or coach' column
# Add legend (legend =True)
leeds.plot(ax=ax, column ='Bus, minibus or coach', legend=True)
# show the map
plt.show()

<font color= 'orchid'> <b>Code your own choropleth map for 'Works mainly from home'<b></font><br>
    Answer at the end of the workbook

In [ ]:
# code your choropleth map here!

### Counts or rates?
The maps above show the *number* of people in each LSOA. Counts like these partly reflect how many people live in each area, so a choropleth map of raw counts can be misleading. It is usually better to map a rate or a percentage.

The travel-to-work columns add up to 'Total_distance_to_work' (the total number of residents counted in these census tables). If we take away the people who are not in employment and those who work mainly from home, we are left with the number of people who travel to work. We can use that as the denominator.

<font color='orchid'><b>Run the code below to calculate and map the percentage of commuters who travel by bus</b></font>

In [ ]:
# number of people who travel to work in each LSOA
commuters = leeds['Total_distance_to_work'] - leeds['Not in employment or aged 15 years and under'] - leeds['Work mainly at or from home']

# percentage of commuters who travel by bus
leeds['pct_bus'] = leeds['Bus, minibus or coach'] / commuters * 100

f,ax = plt.subplots(1, figsize=(16,8))
# plot the percentage, and label the colour bar
leeds.plot(ax=ax, column='pct_bus', legend=True, legend_kwds={'label': '% of commuters travelling by bus'})
ax.set_axis_off()
ax.set_title('Percentage of commuters who travel to work by bus (LSOA)')
plt.show()

### Plotting Categorical variables
First let's generate a categorical variable. We might be interested in the most common way of travelling to work, other than by car, in each LSOA.

Have a look at the <code>.idxmax()</code> documentation: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.idxmax.html

- what are we doing here?
- what does <code>axis=1</code> do in the function?


Comment the code below to demonstrate you know what it is doing

In [ ]:
leeds['most_common_non_car_mode']=leeds[['Underground, metro, light rail, tram',
       'Train', 'Bus, minibus or coach', 'Taxi',
       'Motorcycle, scooter or moped', 'Bicycle', 'On foot']].idxmax(axis=1)

It is possible to create choropleth maps with categorical variables (e.g. most_common_non_car_mode). To do this we specify <code> categorical=True,</code> within the <code>plot()</code> function.<br> 

Note the legend of the map is now separate colours and not a continuous colour bar

<font color ='orchid'> <b> Run the code below to plot a categorical choropleth </font>

In [ ]:
f,ax = plt.subplots(1, figsize=(16,8))
# plot the 'most_common_non_car_mode' column as categories
leeds.plot(ax=ax, column='most_common_non_car_mode', categorical=True, legend=True)
plt.show()

 Let's improve this visualisation a bit.

In [ ]:
f,ax = plt.subplots(1, figsize=(16,8))
# plot the 'most_common_non_car_mode' column
# use the 'Set2' palette, which is designed for categorical data
leeds.plot(ax=ax, column ='most_common_non_car_mode', categorical=True, cmap='Set2', legend=True)
# get the axis for the legend
leg = ax.get_legend()
# move the legend towards the bottom left corner
leg.set_bbox_to_anchor((0.3, 0.2))
# remove axis
ax.set_axis_off()
# set title
ax.set_title('Most common way of commuting, other than by car (LSOA)')
# show plot
plt.show();

<code>leg.set_bbox_to_anchor((0.3, 0.2))</code> Try editing the coordinates in the <code>set_bbox_to_anchor()</code> function. See where the legend moves.

In [ ]:
# list the wards in Leeds (we will use one of these in the next section)
leeds['WD24NM'].unique()

## Sub-setting spatial data
We can also work with and plot just a subset of the spatial data. For example we might only be interested in the Ward of 'Headingley & Hyde Park' in Leeds. In which case we can use the <code>.loc[]</code> function to locate all rows (LSOAs) where the ward is 'Headingley & Hyde Park'. Just like we would normally do in a non-spatial pandas data frame.<br>

<font color ='orchid'> <b>Run the code below </font>

In [ ]:
# locate LSOAs (rows) in the Ward ('WD24NM') of 'Headingley & Hyde Park'
hyde_park = leeds.loc[leeds['WD24NM']== 'Headingley & Hyde Park',:]
# view the newly created hyde park dataframe
hyde_park

It is now possible to plot just Hyde Park and Headingley. We can also put maps side by side to compare them. Here we are going to compare the number of people commuting by bike and on foot in each LSOA.<br>

<font color = 'orchid'> <b> Run the code below,</b> make sure you understand what each line does (there is a lot going on)</font>

In [ ]:
# create a figure with two subplots (maps)
f,ax = plt.subplots(1,2, figsize=(12,6))

# plot the number commuting on foot in subplot 1
hyde_park.plot(ax=ax[0], column ='On foot', legend=True)

# plot the number commuting by bicycle in subplot 2
hyde_park.plot(ax=ax[1], column ='Bicycle', legend=True)

# give subplot 1 an informative title
ax[0].set_title('On foot')

# give subplot 2 an informative title
ax[1].set_title('Bicycle')

# make axis invisible for subplot 1
ax[0].set_axis_off()

# make axis invisible for subplot 2
ax[1].set_axis_off()

# show figure
plt.show()

In [ ]:
## Let's improve the plots further

# create a figure with two subplots (maps)
f,ax = plt.subplots(1,2, figsize=(12,6))

# To compare the two maps fairly, they need to use the same colour scale.
# We do this by giving both plots the same minimum (vmin) and maximum (vmax) values.

# minimum = 0 (as these are counts)
vmin = 0
# calculate the maximum value across both columns
vmax = hyde_park[['On foot','Bicycle']].max().max()

# plot the number commuting on foot in subplot 1,
# shrink the legend and use the shared colour scale
hyde_park.plot(ax=ax[0], column ='On foot', legend=True, legend_kwds={'shrink': 0.4}, vmin=vmin, vmax=vmax)

# plot the number commuting by bicycle in subplot 2,
# shrink the legend and use the shared colour scale
hyde_park.plot(ax=ax[1], column ='Bicycle', legend=True, legend_kwds={'shrink': 0.4}, vmin=vmin, vmax=vmax)

# give subplot 1 an informative title
ax[0].set_title('Number of working population commuting on foot')

# give subplot 2 an informative title
ax[1].set_title('Number of working population commuting by bicycle')

# make axis invisible for subplot 1
ax[0].set_axis_off()

# make axis invisible for subplot 2
ax[1].set_axis_off()

# show figure
plt.show()

## Subplots and axes

A quick note on the subplot and ax indexing. It can be a little tricky getting your head around the indexing used when creating subplots.
To create multiple subplots you use the code below, specifying firstly the number of rows then the number of columns. <br>
<code>.subplots([number of rows], [number of columns])</code>

When you are then specifying each individual plot, you need to specify the axes of the plot, using the code <code> ax=ax[]</code>. The indexing for the axes starts at 0. I.e. the first row is 'row 0' and the first column, 'column 0'. This is illustrated below.

![](https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/blob/main/notebooks/screenshots/axes_diagram.png?raw=true)

For a set of subplots with just one row you only need to specify the column in the <code>ax=ax[]</code> function. <br>
I.e. <code>ax=ax[column number]</code> <br>

I.e.<code> f, ax = plt.subplots(1, 2) <br>
dataframe.plot(ax=ax[0], ...
dataframe.plot(ax=ax[1], ... </code>
![](https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/blob/main/notebooks/screenshots/axes_2.png?raw=true)

Similarly for a set of subplots with just one column you only need to specify the row in the <code>ax=ax[]</code> function. <br>
<code> f, ax = plt.subplots(2, 1) <br>
dataframe.plot(ax=ax[0], ...
dataframe.plot(ax=ax[1], ... </code>
![](https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/blob/main/notebooks/screenshots/axes_3.png?raw=true)

For a set of subplots with multiple rows and columns you need to specify both the row and the column (row first, then column): <code>ax=ax[row, column]</code><br>
<code>f, ax = plt.subplots(2, 3)<br>
dataframe.plot(ax=ax[0,0], ...
dataframe.plot(ax=ax[0,1], ...
dataframe.plot(ax=ax[0,2], ...
dataframe.plot(ax=ax[1,0], ...
dataframe.plot(ax=ax[1,1], ...
dataframe.plot(ax=ax[1,2], ...</code>

![](https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/blob/main/notebooks/screenshots/axes_4.png?raw=true)

<div class="alert alert-block alert-warning">
    
## Independent coding task: Using USA data

### Tasks:
- check the CRS of both geo data frames
- produce a layered map of cities and states
- produce a choropleth map of state population (the 'population' column)
- produce a map of the cities, coloured by elevation ('ELEV_IN_FT')
- colour the cities by state
- subset the data to plot a single state e.g. Washington

### The data

In [ ]:
# read the USA data directly from GitHub (these are example datasets from the geoplot package)
usa_cities = gpd.read_file('https://raw.githubusercontent.com/ResidentMario/geoplot-data/master/usa-cities.geojson')
contiguous_usa = gpd.read_file('https://raw.githubusercontent.com/ResidentMario/geoplot-data/master/contiguous-usa.geojson')

# remove cities in states not in the contiguous USA (not connected directly to the mainland), for ease of plotting
continental_usa_cities = usa_cities.loc[(usa_cities['STATE'] !="HI") & (usa_cities['STATE'] !="AK" ) & (usa_cities['STATE'] !="PR")]


In [ ]:
continental_usa_cities.head()

In [ ]:
continental_usa_cities.plot()

In [ ]:
contiguous_usa.head()

In [ ]:
contiguous_usa.plot()

In [ ]:
# Add more cells of code here

# Part B: Manipulating spatial data

## Geometric Manipulations

In [ ]:
#For the sake of this exercise we are going to change the projection of the data
bus_stops_p1 =bus_stops.to_crs('epsg:4326')


Let's compare CRS

In [ ]:
bus_stops.crs

Notice that in the British National Grid (EPSG:27700) the axis information indicates the units used are metres: <br>

<code>Axis Info [cartesian]:</code><br>
<code> - E[east]: Easting (metre)</code><br>
<code> - N[north]: Northing (metre)</code>


In [ ]:
bus_stops_p1.crs


Notice that in the WGS 84 (World Geodetic System) geographic CRS (EPSG:4326) the axis information indicates the units used are degrees: <br>

<code>Axis Info [ellipsoidal]:</code><br>
<code>- Lat[north]: Geodetic latitude (degree)</code><br>
<code>- Lon[east]: Geodetic longitude (degree)</code><br>


The CRS therefore matters when we come to do spatial analysis with our data (just like in every other GIS)

### Buffers

A buffer in a geographic information system (GIS) is a zone around a map feature, measured in units of distance.

In Python we specify the buffer size: the radius of the buffer (in this case it is measured in metres as the CRS is EPSG:27700).

E.g.:

<code> dataframe.buffer(distance =800) </code> distance = 800 metres.

800 metres is often used to represent a 10-minute walk (400 metres is roughly a 5-minute walk).

In [ ]:
# Plot 800 m buffers around bus stops

# plot one subplot (1 map), with dimensions 16 X 8
f, ax = plt.subplots(1, figsize=(16, 8))

bus_stops.buffer(distance=800).plot(ax=ax)

plt.show()

If the data were in EPSG:4326 the buffer distance would be in degrees. Run the code below and see what happens (geopandas also prints a warning that the CRS is geographic). We only plot the first five bus stops: one degree of latitude is about 111 km, so each 10-degree buffer is enormous!

In [ ]:
bus_stops_p1.head(5).buffer(distance=10).explore()

In [ ]:
# run the code without plotting- notice we generate an array of geometries 
bus_stops.buffer(distance=800)

In [ ]:
# we can check the data type- a geopandas array
bus_stops.buffer(distance=800).dtype

In order to keep information about the area being buffered in the newly created buffer geo-data-frame it is useful to copy the original data you want to conduct the geometric manipulation on, and name it something else e.g. dataframe_buffer.

Then replace the geometry column in the copied data with the calculated buffer geometry.

E.g.
<code> dataframe_buffer = dataframe.copy() </code>

<code> dataframe_buffer['geometry'] = dataframe.buffer(distance)</code>

In [ ]:
# copy the bus stops dataframe
bus_stops_buffer = bus_stops.copy()

# apply the function (replacing the geometry column with the buffer geometry)
bus_stops_buffer['geometry'] = bus_stops.buffer(distance=800)

bus_stops_buffer.head()

Note because we copied the bus_stops data frame above the bus_stops_buffer data frame still contains the name of the bus stop and other information e.g. average number of buses in the morning.


We can add the bus_stop_buffer layer to the map along with the bus_stop and leeds layers.

<font color = 'orchid'> <b> Run the code below, read the comments to understand what each line of code is doing  </font>

In [ ]:
# plot one subplot (1 map), with dimensions 16 X 8
f, ax = plt.subplots(1, figsize=(16, 8))

# define the basemap plot it on the subplot axis
base = leeds.plot(ax=ax, color='grey')

# plot the bus stop buffers on the basemap axis, colour buffers blue
bus_stops_buffer.plot(ax=base,color='blue', alpha=0.2)

# plot the bus stops on the basemap axis, colour the bus stops red
bus_stops.plot(ax=base,color='red', markersize=1)

# show the map
plt.show()

### Centroids

Put simply, the centroid is the geometric centre of a polygon. For irregular shapes the centroid can fall outside the polygon. If you need a point that is guaranteed to be inside, use <code>.representative_point()</code> instead.

#### Why calculate centroids?

Lots of geometric manipulations and analysis use centroids. For example you might use centroids as a proxy to measure the distance between two polygons.


<font color = 'orchid'> <b>Use the code </b> <code> dataframe.centroid </code> <b>to find the LSOA centroids (run the next 3 cells of code for the worked example below) </font>

In [ ]:
# copy the leeds dataframe
leeds_centroid = leeds.copy()

# calculate the centroid
# and replace the leeds geometry (polygon) with the centroid geometry (point)
leeds_centroid['geometry'] = leeds.centroid

# check - geometry should contain point data
leeds_centroid.head()

In [ ]:
# plot to check it looks as expected
leeds_centroid.plot();

In [ ]:
# plot one subplot (1 map), with dimensions 16 X 8
f, ax = plt.subplots(1, figsize=(16, 8))

# define the basemap plot it on the subplot axis
base = leeds.plot(ax=ax, color='grey')

# plot the lsoa centroids on the basemap axis, colour the centroids purple
leeds_centroid.plot(ax=base,color='purple')

# show the map
plt.show()

We can also change marker size and colour of plotted points to reflect the data they represent (as we did with the choropleth maps for polygon data).

<font color = 'orchid'> <b>Run the code below</font>

In [ ]:
# plot one subplot (1 map), with dimensions 16 X 8
f, ax = plt.subplots(1, figsize=(16, 8))

# define the basemap plot it on the subplot axis
base = leeds.plot(ax=ax, color='grey')

# plot the lsoa centroids on the basemap axis, colour and size the centroids based on number of people commuting by bus
leeds_centroid.plot(ax=base, markersize='Bus, minibus or coach', column='Bus, minibus or coach',  legend=True)

# make axis invisible
ax.set_axis_off()

# show the map
plt.show()

We can improve this visualisation further by scaling the marker size, adjusting the legend extent and adding a title

<font color = 'orchid'> <b>Run and read the code below, check you understand what is happening on each line</font>

In [ ]:
# plot one subplot (1 map), with dimensions 16 X 8
f, ax = plt.subplots(1, figsize=(16, 8))

# define the basemap plot it on the subplot axis
base = leeds.plot(ax=ax, color='grey')

# plot the lsoa centroids on the basemap axis, colour and size the centroids based on number of people commuting by bus
# scale the marker size to fit better and make them semi-transparent to see overlap in dense areas
# start colorbar legend at 0 as count data
leeds_centroid.plot(ax=base, markersize=leeds_centroid['Bus, minibus or coach']/4, column='Bus, minibus or coach', alpha=0.8, legend=True, vmin=0)

# make axis invisible
ax.set_axis_off()

# add title, increase fontsize
ax.set_title('Number of people commuting by bus in Leeds by LSOA', fontsize =14)

# show the map
plt.show()

### Convex hull polygons

A convex hull is the smallest polygon that you can draw around a collection of points/a polygon.

<font color = 'orchid'> <b>Run the code below to create convex hull polygons (using the code </b><code> dataframe.convex_hull</code><b> around the LSOAs in Leeds </font>

In [ ]:
leeds_convex_hull = leeds.copy()

# replace the geometry column with the calculated convex hull polygon geometries 
leeds_convex_hull['geometry'] = leeds.convex_hull

# plot one subplot (1 map), with dimensions 16 X 8
f, ax = plt.subplots(1, figsize=(16, 8))

# define the basemap plot it on the subplot axis
base = leeds.plot(ax=ax, color='grey')

# plot the convex hull polygons
# .boundary allows us to just plot the polygon outline
leeds_convex_hull.boundary.plot(ax=base)

# make axis invisible
ax.set_axis_off()

# show the map
plt.show()

## Part C: Spatially aggregating data
Often our data will not be at the same spatial scale, so we may need to aggregate areas of data together to get them to the same spatial scale. Or we may only be interested in larger spatial trends. Therefore, we need to convert our smaller area data to larger area data. In geopandas we can easily do this using the <code>dissolve</code> function.
In this example we are going to aggregate LSOAs up to ward (WD24NM) level. <br>
Think of dissolve as removing all the internal LSOA borders within the ward to leave just the ward outline.<br>
The data for the ward also gets aggregated, below we have aggregated all the numeric data <br>
<font color='orchid'> <b> Run the code below </font>

In [ ]:
# specify which columns from the leeds dataframe we are going to aggregate: 
# #this should include the column we are going to aggregate the data by (i.e. WD24NM), the geometry column and any other columns we want to aggregate
# use the WD24NM column to inform the dissolve, 
# in this case sum is a suitable aggregation to get the total within the wards, we could also use 'mean' to get the average within the wards
leeds_wards = leeds[['WD24NM','Total_distance_to_work', 'Less than 10km',
       '10km to less than 30km', '30km and over', 'Works mainly from home',
       'Not in employment or works mainly offshore, in no fixed place or outside the UK',
       'Work mainly at or from home', 'Underground, metro, light rail, tram',
       'Train', 'Bus, minibus or coach', 'Taxi',
       'Motorcycle, scooter or moped', 'Driving a car or van',
       'Passenger in a car or van', 'Bicycle', 'On foot',
       'Other method of travel to work',
       'Not in employment or aged 15 years and under','geometry']].dissolve(by='WD24NM', aggfunc='sum').reset_index()

# view the new leeds_wards dataframe
leeds_wards

In [ ]:
# plot the new leeds_wards df

leeds_wards.explore(column ='Bus, minibus or coach', vmin=0)

Think carefully about how you want to aggregate the columns. Counts (like the numbers here) can be summed, but percentages or rates cannot. We can also assign a different aggregation function to each column (as we did with <code>.agg()</code> when looking at the bike share data).

E.g.
<code>, aggfunc={'Bicycle':'sum', 'On foot':'mean'})</code>

With a large dataset you might want to consider which columns you include in the aggregated spatial data frame.


#### Replicate and edit the code above to aggregate the LSOA data to plot mean number of people commuting by bike per ward 
(Answer at the end)

In [ ]:
# Write your code here

## Part D: Spatial Data Joins 
## Joining a non-spatial dataset to a spatial dataset

We are going to read in some data about the resident population of LSOAs across England and Wales and join it to the Leeds data frame

<font color ='orchid'><b> Run the code below to read in a csv of LSOA level resident population data </font>

In [ ]:
# read in residential population data from 2021 census
lsoa_resident_pop =pd.read_csv('https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_5/lsoa_resident_pop.csv')
# use loc to subset data to only contain data for Leeds
leeds_resident_pop =lsoa_resident_pop.loc[lsoa_resident_pop['geography'].str.contains('Leeds'),:]
# have a quick look at the dataframe
leeds_resident_pop.head()

In [ ]:
print('Check both datasets have the same number of rows:', leeds_resident_pop.shape[0], leeds.shape[0])

In [ ]:
leeds.head()

The 'LSOA21CD' column in the Leeds data frame matches that of the 'geography code' column in the leeds_resident_pop data frame. As they are formatted in exactly the same way and each row is a unique LSOA, the geography code/ LSOA21CD is a unique identifier common to both data frames. Therefore, we will use this column to join our datasets. <br>
The code we will use is <code>pd.merge()</code>. We need to specify a few parameters within the function:<br>

- Firstly we specify which data frames we want to join (in the order we want to join them).<br>
<code> pd.merge(leeds, leeds_resident_pop... </code> <br>


- Secondly we need to specify the column in each data frame, leeds is on the left so we specify left_on='LSOA21CD': as we are using the 'LSOA21CD' column from the leeds dataframe. And the leeds_resident_pop data frame is on the right, so we specify right_on='geography code' as we are using the 'geography code' column for our join.<br>
<code> pd.merge(leeds, leeds_resident_pop, left_on='LSOA21CD', right_on='geography code' ... </code> <br>


- Finally, we need to specify how the tables are joined. This is based on SQL join types (see the [pandas guide to merging](https://pandas.pydata.org/docs/user_guide/merging.html)). In this case we are using a left join (we keep all the data in the left data frame and just add the columns from the right dataframe on the end).
<code> pd.merge(leeds, leeds_resident_pop, left_on='LSOA21CD', right_on='geography code', how='left') </code> <br>

<b>Note:</b> put the spatial data frame (leeds) on the left. The result takes its type from the left-hand data frame, so if the non-spatial data frame were on the left the result would be an ordinary DataFrame rather than a GeoDataFrame, and <code>.plot()</code> would not draw a map.

In [ ]:
leeds_pop = pd.merge(leeds, leeds_resident_pop, left_on='LSOA21CD', right_on='geography code', how='left')
leeds_pop.head()

In [ ]:
# check if any of the rows in the newly joined columns are NA, if so this might suggest that the join has not worked as expected 
leeds_pop['Residence type: Total; measures: Value'].isna().any()

Now we will visualise our newly joined data by plotting the total resident population in each LSOA

In [ ]:
f,ax = plt.subplots(1, figsize=(16,8))
leeds_pop.plot(column='Residence type: Total; measures: Value',legend=True,ax=ax)
# remove axis
ax.set_axis_off()
plt.show();

With a continuous colour scale it can be difficult to tell areas apart, especially when a few areas have much larger values than the rest. To make a more informative map we can group the values into classes using <code>scheme=' '</code>, and set the number of classes with <code>k</code>.
Common schemes include <code>['equal_interval', 'quantiles', 'natural_breaks', 'fisher_jenks']</code>. With 'quantiles', each class contains the same number of LSOAs. (These schemes come from the mapclassify package that we installed at the start.)

In [ ]:
f,ax = plt.subplots(1, figsize=(16,8))
# classify the data into 5 quantiles, show the class limits as whole numbers in the legend,
# and draw thin grey boundaries so the palest areas are still visible
leeds_pop.plot(column='Residence type: Total; measures: Value',legend=True, cmap='YlGn',scheme='quantiles', k=5, legend_kwds={'fmt': '{:.0f}'}, edgecolor='lightgrey', linewidth=0.3, ax=ax)
# remove axis
ax.set_axis_off()
# position legend
ax.get_legend().set_bbox_to_anchor((.12, .4))
# add title
ax.set_title('Leeds total resident population (LSOA)', fontsize=15)
# add a North arrow (look at the documentation here: https://github.com/pmdscully/geo_northarrow)
add_north_arrow(ax=ax, scale=.75, xlim_pos=0.1, ylim_pos=.85, color='#000', text_scaler=2, text_yT=-1.25)
plt.show();

<font color='orchid'><b> Write your own code to map 'Residence type: Lives in a communal establishment; measures: Value'</font> <br>
Answer at the end

In [ ]:
# plot here!





## Spatially Joining Data

It is also possible to join data based on their spatial relationship to each other using <code>gpd.sjoin()</code>.
For example we might want to know the LSOA each bus stop is in.


Like with <code>pd.merge()</code> there are parameters we need to specify when using <code>gpd.sjoin()</code>

- Again firstly we specify which data frames we want to join (in the order we want to join them).<br>
<code>gpd.sjoin(bus_stops, leeds, ... </code> <br>


- Secondly we need to specify how the tables are joined: 'inner', 'left' or 'right' (again based on SQL join types). In this case we are using an inner join, so only the bus stops that fall within an LSOA in Leeds are kept.
<code> gpd.sjoin(bus_stops, leeds, how="inner",... </code> <br>

- Finally, we need to specify the spatial relationship (the 'predicate') that decides whether two features are joined. These are the relationships illustrated in the lecture slides. The most common are:
    - <b>intersects</b>: the features share any part of space (they touch or overlap). This is the default.
    - <b>within</b>: the feature in the left data frame lies entirely inside the feature in the right data frame.
    - <b>contains</b>: the feature in the left data frame completely contains the feature in the right data frame.

  Others include 'touches', 'crosses' and 'overlaps'. See the [geopandas documentation](https://geopandas.org/en/stable/docs/user_guide/mergingdata.html) for the full list.

<font color = 'orchid'><b> Run the code below to spatially join the data </font>

In [ ]:
bus_stops_leeds = gpd.sjoin(bus_stops, leeds, how="inner", predicate='intersects')
# look at the data: notice that we now have the LSOA level info for each bus stop 
bus_stops_leeds.head()

Note when mapped we now just have bus stops within Leeds, not the whole of West Yorkshire

In [ ]:
# let us plot the bus stops now coloured by the number of people commuting by bus in that LSOA
f,ax = plt.subplots(1, figsize=(16,8))

# map the leeds LSOAs (basemap)
base = leeds.plot(ax=ax, color='grey')
# plot the bus stops and colour them based on n of people commuting by bus in the LSOA in which the bus stop is
bus_stops_leeds.plot(ax=base,  column ='Bus, minibus or coach', cmap='viridis',markersize=2, vmin=0,legend=True)
# remove axis
ax.set_axis_off()
plt.show()

#### Overlay - creating spatial layers from the intersections, unions, and differences between maps
I have included this for reference with a worked example, to show you how we can look at where spatial data overlaps. A list of the overlay operations can be found here: https://geopandas.org/en/stable/docs/user_guide/set_operations.html
Have a read and explore the data.

In [ ]:
# get the intersection between the bus stop buffers (a 10-minute walk) and the LSOAs,
# to see which parts of each LSOA are within a 10-minute walk of a bus stop
lsoas_in_buff = gpd.overlay(bus_stops_buffer, leeds, how='intersection')
# look at the data
lsoas_in_buff

In [ ]:
# plot the results
f,ax = plt.subplots(1, figsize=(16,8))

# lsoa base map
base = leeds.plot(ax=ax,color='grey')
# plot the intersection of bus stop buffer and lsoas 
lsoas_in_buff.plot(ax=base, alpha=0.8)
# remove axis
ax.set_axis_off()
plt.show();

## Part E: Kernel density maps and basemaps
### Kernel density map

Much like the kernel density plots we produced in Week 4, this produces a kernel density estimate of spatial point data: a smooth surface showing where the points are concentrated. The <code>bw_adjust</code> parameter of <code>sns.kdeplot()</code> controls how smooth the surface is.

In [ ]:
## Seaborn example

f,ax = plt.subplots(1, figsize=(16,8))

# map the leeds (basemap)
base = leeds.plot(ax=ax, color='grey')

# map the kernel density estimate of bus stops
sns.kdeplot(bus_stops_leeds, x='x', y='y', color='Red', fill=True,ax=base, alpha=0.8)

# remove axis
ax.set_axis_off()

plt.show()

### Adding base maps
Base maps provide the reader with context for a map. When zoomed in further, the basemap may show road networks or points of interest.

We are going to use contextily to add the background map to the geographic data using the <code>add_basemap()</code> function.

#### Aligning the CRS
But first we need to convert the CRS of the data we want to plot to Web Mercator (EPSG:3857), so that the base maps and the geographic data we are plotting align.

Unless the data file I want to plot is particularly big I tend to save the data with the Web Mercator CRS as a new geodataframe to avoid confusion.

In [ ]:
leeds.crs

In [ ]:
# convert CRS to epsg=3857 Web Mercator
leeds_WM = leeds.to_crs(epsg=3857)

fig,ax = plt.subplots(1, figsize=(10,10))

# plot the Leeds data as usual
leeds_WM.plot(column='Works mainly from home',ax=ax,alpha=0.5)

# add ctx basemap to ax (by default contextily uses OpenStreetMap Humanitarian tiles)
ctx.add_basemap(ax, crs=leeds_WM.crs)

ax.set_axis_off()

#### Different base maps.
By default <code>ctx.add_basemap()</code> uses OpenStreetMap Humanitarian tiles.

To see the other available base maps we can get the provider keys:

List of contextily providers:

In [ ]:
ctx.providers.keys()

Each will have their own options, accessed by the code: <code> ctx.providers.<font color='red'>provider_from_above</font>.keys()</code>

E.g. <code> ctx.providers.Esri.keys()</code>

Some providers (for example CartoDB and Stadia) now require a free API key. You can check with, e.g., <code>ctx.providers.CartoDB.Voyager.requires_token()</code>

In [ ]:
ctx.providers.Esri.keys()

We can then use the provider and provider options to choose our base map e.g. <code>Esri.WorldGrayCanvas</code>

In the first example below we plot the data in its original CRS (British National Grid) and pass <code>crs=leeds.crs</code> to <code>add_basemap()</code>. This tells contextily to reproject the map tiles to match our data (the tiles can look slightly blurred as a result). In the second example we use the Web Mercator version of the data, so we do not need to give a <code>crs</code>.

In [ ]:


f,ax = plt.subplots(1, figsize=(16,16))

# plot the Leeds data as usual
leeds.plot(alpha=0.5,column='Works mainly from home',ax=ax)

# add ctx basemap to ax, specifying the basemap provider and options
ctx.add_basemap(ax,source=ctx.providers.Esri.WorldGrayCanvas, zoom=12, crs=leeds.crs)

ax.set_axis_off()


In [ ]:
f,ax = plt.subplots(1, figsize=(16,16))

# plot the Leeds data as usual
leeds_WM.plot(alpha=0.5,column='Works mainly from home',ax=ax)

# add ctx basemap to ax, specifying the basemap provider and options
ctx.add_basemap(ax,source=ctx.providers.Esri.WorldImagery, zoom=12)

ax.set_axis_off()

<font color='orchid'><b>  Play around with the providers and options to get different base maps.

# Stretch Activity: Spatial analysis in practice with the Priority Places for Food Index

In this final part of the practical we will put some of the techniques that you learned in parts A-E into practice exploring the Priority Places for Food Index. That index consists of a number of different 'domains' (the individual variables that are combined to make an overall index) and we will map and analyse them as well. 

## Data Exploration and Preparation
### Read in the Priority Places for Food Index (PPFI) data
It is stored in the [data/week_5](https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/tree/main/data/week_5) directory.

In [ ]:
PPFI = pd.read_csv('https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_5/ppfi_index_v2_nov2023.csv')

In [ ]:
PPFI.head()

Write a few lines of code to explore the data

In [ ]:
# explore data types

### Read in the spatial dataframe

The PPFI data are _a-spatial_ (they have no spatial information included). However, the column `lsoa21cd` contains the unique LSOA code for the neighbourhood that each row in the dataset represents, so we can use this code to link the PPFI to a spatial dataset. 

Begin by reading in a spatial LSOA file for Leeds. Note that we use the `gpd` (geopandas) package to load the data. Geopandas is an extension to pandas that provides lots of useful functions for working with spatial data. The most commonly used aspect of geopandas is the `GeoDataFrame`. This is like a normal pandas DataFrame, but has some additional functionality for working with spatial data.

In [ ]:
leeds_shp = gpd.read_file('https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_5/Leeds.geojson')

In [ ]:
leeds_shp.head()

**There are two important things to note in the table above:**
 - Can you see the `LSOA21CD` column? That contains the LSOA code for each neighbourhood and we can use that to link the boundary data to the PPFI data
 - Did you notice the column called `geometry`? That is a special column that all GeoDataFrames have and stores the _spatial_ information (in our case it is a list of coordinates that make up each LSOA's polygon). This allows us to draw maps of the spatial data and do more advanced GIS analysis.

I have included the code below for generating the Leeds.geojson file, for reference for your final project.
We will talk more about finding and downloading data in the coming weeks

In [ ]:
# # Data downloaded from https://geoportal.statistics.gov.uk/maps/761ecd09b4124843b95511a242e2b1a1
# shp =gpd.read_file('/Users/fran/Downloads/Lower_layer_Super_Output_Areas_2021_EW_BFE_V9_-3647710721716634062.geojson')
# leeds_shp =shp.loc[shp['LSOA21NM'].str.contains('Leeds'),:]
# leeds_shp.to_file('Leeds.geojson')

### Explore, the spatial dataframe

(Hint, the `.explore()` function which is available on all GeoDataFrames is useful here...)

In [ ]:
# Write code to explore the data spatially


### Join the PPFI data to the spatial dataframe of Leeds

We use the `merge` function and use the LSOA code to join the neighbourhoods.

In [ ]:
# join the PPFI data to the leeds_shp geodataframe using a lefthand join, on the common ID 'LSOA21CD'/'lsoa21cd'
leeds_PPFI =leeds_shp.merge(PPFI, how='left',left_on='LSOA21CD',right_on='lsoa21cd')

In [ ]:
# Now we have the data from the PPFI attached to the LSOA boundaries
leeds_PPFI.columns

In [ ]:
# map to check it has worked

# look at where is higher risk of Food Insecurity, don't forget smaller numbers = 'higher priority'
leeds_PPFI.explore('pp_dec_combined', cmap='Reds_r')

### Mapping the Priority Places for Food Index Deciles in Leeds

In [ ]:
# create a list of the columns we want to plot
decile_cols =['pp_dec_domain_supermarket_proximity',
       'pp_dec_domain_supermarket_accessibility',
       'pp_dec_domain_ecommerce_access', 'pp_dec_domain_socio_demographic',
       'pp_dec_domain_nonsupermarket_proximity',
       'pp_dec_domain_food_for_families', 'pp_dec_domain_fuel_poverty']

# create a for loop for plotting a map of each domain
# this saves writing out 7 blocks of code for each visualisation

# for each item in a range from 0 to 7 (number of items in the decile_cols list)
for i in range (0, len(decile_cols)):

    # produce a plot
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))

    # get the ith item in the decile column list and plot
    leeds_PPFI.plot(column=decile_cols[i],


    #### format the plot ###

    # reduce linewidth between polygons
    linewidth =0.1,
    # specify data is categorical (ordinal)
    categorical=True,
    # show the legend
    legend=True,
    # define the legend palette
    cmap='RdBu',
    # use the defined axis
    ax=ax,
    # position the legend
    legend_kwds={'loc': 'center left','bbox_to_anchor':(1,0.5)})

    # add a title based on the column plotted, formatting the title to look better
    plt.title(decile_cols[i].replace('_',' ').replace('pp dec domain ','').capitalize()+' domain in Leeds')

    # do not plot with the axis showing
    plt.axis('off')

    # save the figure as an image with name reflecting the domain plotted
    plt.savefig(str(decile_cols[i])+'_'+'Leeds'+'.jpg',bbox_inches='tight');

### Explore the association between PPFI domains
To better understand how the domains that make up the 'combined' PPFI index are associated we can plot a pair plot (each domain against the other domains)

In [ ]:
# plot pairplot, adjust height so domain names fit on axis
sns.pairplot(leeds_PPFI[['domain_supermarket_proximity',
       'domain_supermarket_accessibility', 'domain_ecommerce_access',
       'domain_socio_demographic', 'domain_nonsupermarket_proximity',
       'domain_food_for_families', 'domain_fuel_poverty']], height=4)

#### Quantify the association between PPFI domains using Spearman's rank correlation

In [ ]:
# Calculate Spearman's rank correlation
ppfi_domains_corr =leeds_PPFI[['domain_supermarket_proximity',
       'domain_supermarket_accessibility', 'domain_ecommerce_access',
       'domain_socio_demographic', 'domain_nonsupermarket_proximity',
       'domain_food_for_families', 'domain_fuel_poverty']].corr(method = 'spearman')
ppfi_domains_corr

#### Visualise the Spearman's rank correlation of the PPFI domains

In [ ]:
# define plot size
fig,ax = plt.subplots(figsize=(8,8))

# define mask to apply to upper right hand corner of the plot
data_to_mask = np.triu(np.ones_like(ppfi_domains_corr))

# define axis tick labels
# get the index and the columns, replace the underscores with spaces and remove 'domain ' from the name
x_axis_labels = ppfi_domains_corr.columns.str.replace('_',' ').str.replace('domain ','')
y_axis_labels = ppfi_domains_corr.index.str.replace('_',' ').str.replace('domain ','')

# Have a look at what the tick labels look like
print('Labels:',x_axis_labels ,y_axis_labels)

# If we want to capitalize each tick label
# for each element in the list of x_axis_labels, capitalize it
x_axis_labels = [element.capitalize() for element in x_axis_labels]
y_axis_labels = [element.capitalize() for element in y_axis_labels]

# look at the new labels
print('Labels with capital letters:',x_axis_labels ,y_axis_labels)


# plot a heatmap of the correlation dataframe
sns.heatmap(ppfi_domains_corr,
            # annotate so spearman's rank correlation values are displayed on the squares
            annot=True,
            # define colourmap
            cmap='RdBu',
            # define value of minimum colour on cbar
            vmin=-1,
            # define value of maximum colour on cbar
            vmax=1,
            # add the mask
            mask=data_to_mask,
            # use the custom tick labels
            xticklabels=x_axis_labels,
            yticklabels=y_axis_labels,
            # add a label to the cbar
            cbar_kws={'label': "Spearman's Rank correlation"},
            # plot on the axis we defined
            ax=ax)

# Set axis labels
ax.set(xlabel="Priority Places for Food Index Domains",
       ylabel="Priority Places for Food Index Domains",
      title ='Priority Places for Food Index Domain Correlation' );
plt.savefig('Priority_Places_for_Food_Index_Domain_Correlation.png')

#### Let's investigate what is going on the visualisation above.
<b>What happens if you exclude 'mask=data_to_mask'?</b>

In [ ]:
# define plot size
fig,ax = plt.subplots(figsize=(8,8))

# plot a heatmap of the correlation dataframe
sns.heatmap(ppfi_domains_corr,
            # annotate so spearman's rank correlation values are displayed on the squares
            annot=True,
            # define colourmap
            cmap='RdBu',
            # define value of minimum colour on cbar
            vmin=-1,
            # define value of maximum colour on cbar
            vmax=1,
            # comment out the mask
            #mask=data_to_mask,
            # use the custom tick labels
            xticklabels=x_axis_labels,
            yticklabels=y_axis_labels,
            # add a label to the cbar
            cbar_kws={'label': "Spearman's Rank correlation"},
            # plot on the axis we defined
            ax=ax)

# Set axis labels
ax.set(xlabel="Priority Places for Food Index Domains",
       ylabel="Priority Places for Food Index Domains",
      title ='Priority Places for Food Index Domain Correlation' );


<b> Run the code below, to test what each part of the code does. </b>

In [ ]:
# np.ones_like: Returns an array of ones with the same shape and type as a given array.
np.ones_like(ppfi_domains_corr)

In [ ]:
#np.triu() returns the Upper triangle of an array

np.triu(np.ones_like(ppfi_domains_corr))

In the function <code>sns.heatmap()</code>, the <code>mask</code> parameter means: 'If passed, data will not be shown in cells where mask is True', i.e. '1'.

So in our <code>np.triu(np.ones_like(ppfi_domains_corr))</code> array, only values where the array is 0 will be shown in the heatmap.

#### Comment out different parts of the code below to test you know what each parameter in the function is doing

Look at the package documentation (https://seaborn.pydata.org/generated/seaborn.heatmap.html) to see if you can add further features to the visualisation

e.g. make the cbar horizontal or increase the line width between heatmap squares

In [ ]:
# a copy of the code is here for you to explore what the different parameters do

# define plot size
fig,ax = plt.subplots(figsize=(8,8))
# plot a heatmap of the correlation dataframe
sns.heatmap(ppfi_domains_corr,
            # annotate so spearman's rank correlation values are displayed on the squares
            annot=True,
            # define colourmap
            cmap='RdBu',
            # define value of minimum colour on cbar
            vmin=-1,
            # define value of maximum colour on cbar
            vmax=1,
            # add the mask
            mask=data_to_mask,
            # use the custom tick labels
            xticklabels=x_axis_labels,
            yticklabels=y_axis_labels,
            # add a label to the cbar
            cbar_kws={'label': "Spearman's Rank correlation"},
            # plot on the axis we defined
            ax=ax)

# Set axis labels
ax.set(xlabel="Priority Places for Food Index Domains",
       ylabel="Priority Places for Food Index Domains",
      title ='Priority Places for Food Index Domain Correlation' );

# Exercise answers

#### Plot Choropleth of number of people per LSOA working from home

In [ ]:
#### Plot Choropleth of number of people per LSOA working from home
# Define plot size
f,ax = plt.subplots(1, figsize=(16,8))
# Plot the LSOAs, specifying to plot the 'Works mainly from home' column
# Add legend (legend =True)
leeds.plot(ax=ax, column ='Works mainly from home', legend=True)
# show the map
plt.show()

#### Replicate and edit the code above to aggregate the LSOA data to plot mean number of people commuting by bike per ward 

In [ ]:
# specify which columns from the leeds data frame we are going to aggregate:
# calculate mean bike commuting per ward
leeds_wards_mean_bike = leeds[['WD24NM','Bicycle', 'geometry']].dissolve(by='WD24NM', aggfunc='mean').reset_index()

# view the new leeds_wards dataframe
leeds_wards_mean_bike.explore('Bicycle', vmin=0)

#### Write your own code to map 'Residence type: Lives in a communal establishment; measures: Value'

Your map might look different depending on the scheme and number of breaks you used!

In [ ]:
f,ax = plt.subplots(1, figsize=(16,8))
leeds_pop.plot(column='Residence type: Lives in a communal establishment; measures: Value',legend=True, cmap='YlGn',scheme='natural_breaks', k=5, legend_kwds={'fmt': '{:.0f}'}, edgecolor='lightgrey', linewidth=0.3, ax=ax)
# remove axis
ax.set_axis_off()
# position legend
ax.get_legend().set_bbox_to_anchor((.12, .4))
# add title
ax.set_title('Leeds population living in a communal establishment (LSOA)', fontsize=15)
# add a North arrow (look at the documentation here: https://github.com/pmdscully/geo_northarrow)
add_north_arrow(ax=ax, scale=.75, xlim_pos=0.1, ylim_pos=.85, color='#000', text_scaler=2, text_yT=-1.25)
plt.show();
